In [2]:
!pip install pandasql


  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26768 sha256=c00ff07fcd1727e5ddac529dca21b9372c4e103200450f66a7c450c7c5aff0db
  Stored in directory: /Users/apple/Library/Caches/pip/wheels/63/e8/ec/75b1df467ecf57b6ececb32cb16f4e86697cbfe55cb0c51f07
Successfully built pandasql


In [1]:
import pandasql as ps
import pandas as pd

In [12]:
customers = pd.read_csv('./Mall_Customers.csv')
customers

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40
...,...,...,...,...,...
195,196,Female,35,120,79
196,197,Female,45,126,28
197,198,Male,32,126,74
198,199,Male,32,137,18


In [13]:
# Давайте выберем из исходной таблицы покупателей, чей возраст не превышает 25 лет и годовой доход составляет 
# не менее 60 тыс. долларов.
yong_wealthy = customers[(customers['Age'] <=25) & (customers['Annual Income (k$)']>=60)]
yong_wealthy

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
95,96,Male,24,60,52
99,100,Male,20,61,49
100,101,Female,23,62,41
105,106,Female,21,62,42
111,112,Female,19,63,54
113,114,Male,19,64,46
114,115,Female,18,65,48
115,116,Female,19,65,50
124,125,Female,23,70,29
132,133,Female,25,72,34


In [17]:
# Присвоим некоторой переменной sql_query запрос на языке SQLite, заключенный в тройные кавычки, а затем 
# передадим эту переменную в функцию sqldf.

sql_query = '''SELECT * FROM customers WHERE Age <= 25 AND "Annual Income (k$)" >= 60'''
young_wealthy_sql = ps.sqldf(sql_query, locals())
print(young_wealthy_sql)

    CustomerID   Genre  Age  Annual Income (k$)  Spending Score (1-100)
0           96    Male   24                  60                      52
1          100    Male   20                  61                      49
2          101  Female   23                  62                      41
3          106  Female   21                  62                      42
4          112  Female   19                  63                      54
5          114    Male   19                  64                      46
6          115  Female   18                  65                      48
7          116  Female   19                  65                      50
8          125  Female   23                  70                      29
9          133  Female   25                  72                      34
10         135    Male   20                  73                       5
11         139    Male   19                  74                      10
12         145    Male   25                  77                 

In [20]:
# Оконные функции. Группировка. Фильтрация.
# Давайте вычислим средний доход и средний возраст среди 7 мужчин и 7 женщин с наиболее высоким уровнем 
# трат (показатель Spending Score). Для этого нам понадобятся оконная функция и группировка с агрегированием.
query = '''SELECT
             Genre as 'Пол',
             AVG("Annual Income (k$)") AS 'Средний годовой доход',
             AVG(Age) AS 'Средний возраст'
         FROM (select *,
                   ROW_NUMBER() OVER (PARTITION BY Genre ORDER BY "Spending Score (1-100)" DESC)row_number
               from customers)tmp
               WHERE row_number <=7
               GROUP BY Genre '''
select = ps.sqldf(query)
select

,Пол,Средний годовой доход,Средний возраст
0,Female,59.714286,31.714286
1,Male,68.571429,29.714286


In [21]:
# Средствами самого Pandas необходимые нам данные можно получить следующим образом:
male = customers [customers['Genre']=='Male'].sort_values('Spending Score (1-100)',ascending = False).head(7)
male =male.rename(columns={'Genre': 'Пол'})

In [22]:
female = customers [customers['Genre']=='Female'].sort_values('Spending Score (1-100)',ascending = False).head(7)
female =female.rename(columns={'Genre': 'Пол'})

In [24]:
male_female = pd.concat([male, female])
male_female = male_female.groupby(['Пол']).mean()
male_female.rename(columns={'Annual Income (k$)': 'Средний годовой доход', 'Age': 'Средний возраст'}, inplace=True)
male_female = male_female[['Средний годовой доход', 'Средний возраст']]  # закрыли скобку

male_female  


,Средний годовой доход,Средний возраст
Пол,,
Female,59.714286,31.714286
Male,68.571429,29.714286
